# Parlay + OpenEnv: `reset` / `step` rollouts and a simple “training” loop

This notebook uses the **OpenEnv-style** WebSocket API (`cmd: reset` → `cmd: step`) via **`ParlayEnvClient`** to run many negotiation episodes, aggregate **rewards** from the live Parlay grader, and **save** a small JSONL of outcomes (optional) plus **plots**.

This is **not** the same pipeline as SFT/GRPO on a static `jsonl` file — here the environment is the live Space (or a local server), so rewards come from **live** `reset` / `step` turns.

- **Default target:** the public [Parlay Space](https://huggingface.co/spaces/sh4shv4t/Parlay). Use a **small** `N_EPISODES` to avoid rate limits.
- **Alternative:** run `python -m parlay_env.server --port 8001` locally and set `BASE_URL` to your server (see config cell).

Protocol: [`openenv.yaml`](../../openenv.yaml).

In [ ]:
# Colab / Jupyter: install WebSocket client + plotting (parlay_env comes from repo clone below)
%pip install -q websocket-client matplotlib numpy tqdm

import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive  # noqa: F401  # optional: drive.mount("/content/drive")

In [ ]:
# If you are not already inside the Parlay repo, clone it (Colab) or chdir to repo root (local)
import os
import sys
import subprocess
from pathlib import Path

REPO_DIR = Path.cwd()
if not (REPO_DIR / "parlay_env" / "client.py").is_file():
    GITHUB = "https://github.com/sh4shv4t/Parlay.git"
    dest = REPO_DIR / "Parlay"
    if not (dest / "parlay_env" / "client.py").is_file():
        subprocess.run(["git", "clone", "--depth", "1", GITHUB, str(dest)], check=True)
    os.chdir(dest)
    REPO_DIR = dest.resolve()
    print("CWD ->", REPO_DIR)
else:
    print("Using repo at", REPO_DIR.resolve())

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

In [ ]:
# --- Service URL: public Space (default) or local OpenEnv server ---
# Local:  uvicorn:  BASE_URL = "http://127.0.0.1:8001"
BASE_URL = "https://huggingface.co/spaces/sh4shv4t/Parlay"

# How many full episodes to run (keep small on the public Space)
N_EPISODES = 5
MAX_STEPS_PER_EPISODE = 25
RANDOM_SEED = 42

# Scenario / persona grid (all combinations cycled)
SCENARIOS = ["saas_enterprise", "hiring_package", "acquisition_term_sheet"]
PERSONAS = ["shark", "diplomat", "veteran"]

# Where to write JSONL (relative to repo root)
OUT_JSONL = "data/openenv_rollouts.jsonl"

import json
import random
from pathlib import Path

Path("data").mkdir(parents=True, exist_ok=True)
random.seed(RANDOM_SEED)

In [ ]:
from __future__ import annotations

import random

from tqdm.auto import tqdm

from parlay_env.client import ParlayAction, ParlayEnvClient

def heuristic_action(obs: dict, rng: random.Random) -> ParlayAction:
    """Map observation to a single ParlayAction (no LLM). Anchors offer near Nash inside ZOPA."""
    zl = float(obs.get("zopa_lower") or 0.0)
    zu = float(obs.get("zopa_upper") or max(zl + 1.0, 1.0))
    nash = float(obs.get("nash_point") or 0.5 * (zl + zu))
    # small jitter: explore the deal zone
    w = 0.85 + 0.1 * rng.random()
    offer = max(zl, min(zu, w * nash + (1.0 - w) * 0.5 * (zl + zu)))
    text = f"We are prepared to work toward {offer:,.0f} if the other terms line up."
    return ParlayAction(utterance=text, offer_amount=offer)

def run_episode(client, scenario_id: str, persona: str, rng: random.Random) -> dict:
    """One OpenEnv episode: reset → step until done or max steps. Returns record dict."""
    obs = client.reset(scenario_id=scenario_id, persona=persona)
    steps: list[dict] = []
    n = 0
    while n < MAX_STEPS_PER_EPISODE:
        done = bool(obs.get("done") or obs.get("episode_done"))
        if done:
            break
        act = heuristic_action(obs, rng)
        obs = client.step(act)
        n += 1
        steps.append(
            {
                "step": n,
                "reward": float(obs.get("reward", 0.0)),
                "cumulative": float(obs.get("cumulative_reward", 0.0)),
            }
        )
        if bool(obs.get("done") or obs.get("episode_done")):
            break
    return {
        "scenario_id": scenario_id,
        "persona": persona,
        "total_steps": n,
        "cumulative_reward": float(obs.get("cumulative_reward", 0.0)),
        "steps": steps,
    }

In [ ]:
# Main “training” loop: many independent episodes, each a fresh WebSocket session (new reset)
results: list[dict] = []
rng = random.Random(RANDOM_SEED)

with ParlayEnvClient(BASE_URL).sync() as client:
    combos = [(s, p) for s in SCENARIOS for p in PERSONAS]
    for i in tqdm(range(N_EPISODES), desc="episodes"):
        s, p = combos[i % len(combos)]
        rec = run_episode(client, s, p, rng)
        rec["split"] = "train"
        rec["prompt"] = f"openenv:{s}:{p}"  # placeholder label for any downstream SFT
        results.append(rec)

print("Episodes complete:", len(results))
for r in results[:3]:
    print(r["scenario_id"], r["persona"], "R=", round(r["cumulative_reward"], 2), "steps=", r["total_steps"])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rews = [r["cumulative_reward"] for r in results]
plt.figure(figsize=(9, 4))
plt.bar(range(len(rews)), rews, color="#2196F3")
plt.xlabel("Episode index")
plt.ylabel("Cumulative reward")
plt.title("OpenEnv rollouts — heuristic policy (reset / step)")
plt.tight_layout()
plt.show()
print("mean:", float(np.mean(rews)), " std:", float(np.std(rews)), " min:", float(np.min(rews)), " max:", float(np.max(rews)))

In [ ]:
# Save JSONL for optional merge into larger SFT/GRPO datasets (one JSON object per line)
with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")
print("Wrote", len(results), "lines ->", Path(OUT_JSONL).resolve())

### Optional: `openenv` `AutoEnv` (same `reset` / `step` pattern)

If you have **`openenv-core`** installed, you can connect by Space id instead of a full URL. This is equivalent to the `ParlayEnvClient` path above; use **one** approach per session.

In [ ]:
# Skip if you do not have openenv-core, or if you already ran the main loop above in the same kernel.
# try:
#     from openenv import AutoEnv
#     with AutoEnv.from_space("sh4shv4t/Parlay").sync() as ac:
#         o = ac.reset(scenario_id="saas_enterprise", persona="diplomat")
#         print("AutoEnv sample keys:", list(o.keys())[:12])
# except Exception as e:
#     print("AutoEnv not used:", e)
print("Uncomment the block above to try AutoEnv when openenv-core is installed.")